In [1]:
import pandas as pd
import yaml
import json
import pyam
# from jedi.plugins.django import mapping


In [2]:
report = pd.read_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model_report.csv')


In [5]:
report

NameError: name 'report' is not defined

In [3]:
report.Unit.fillna('', inplace=True)
unprocessed = pyam.IamDataFrame(report)
sankey = unprocessed.copy()
sankey = sankey.filter(variable=['in|*', 'out|*'])
sankey = sankey.rename(unit={'': 'GW'}).convert_unit('GW', 'TWh/yr')
all_flows = sankey.timeseries().reset_index()
# Split the strings in the identified variables for further processing
splitted_vars = [v.split('|') for v in all_flows.variable]

# Create auxilary dataframes for processing
aux1_csv = pd.DataFrame(splitted_vars, columns=['flow_type', 'level', 'commodity', 'technology', 'mode']) 
aux2_csv = pd.concat([all_flows.reset_index(drop=True), aux1_csv.reset_index(drop=True)], axis=1)
years = sankey.timeseries().columns
regions = sankey.region

/var/folders/kp/fywbc9rj4lld_vk6pm6m71g00000gn/T/ipykernel_26302/3913715892.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  report.Unit.fillna('', inplace=True)


In [4]:
aux2_csv

,model,scenario,region,variable,unit,unnamed: 0,2010,2015,2020,2025,...,2040,2045,2050,2055,2060,flow_type,level,commodity,technology,mode
0,MESSAGEix-CA,baseline,Alberta|Alberta,in|cooling|cooling__bio_hpl|bio_hpl|M1,TWh/yr,2722,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,in,cooling,cooling__bio_hpl,bio_hpl,M1
1,MESSAGEix-CA,baseline,Alberta|Alberta,in|cooling|cooling__bio_istig_ccs|bio_istig_cc...,TWh/yr,2723,NaN,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,in,cooling,cooling__bio_istig_ccs,bio_istig_ccs,M1
2,MESSAGEix-CA,baseline,Alberta|Alberta,in|cooling|cooling__bio_istig|bio_istig|M1,TWh/yr,2724,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,in,cooling,cooling__bio_istig,bio_istig,M1
3,MESSAGEix-CA,baseline,Alberta|Alberta,in|cooling|cooling__bio_ppl|bio_ppl|M1,TWh/yr,2725,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,in,cooling,cooling__bio_ppl,bio_ppl,M1
4,MESSAGEix-CA,baseline,Alberta|Alberta,in|cooling|cooling__coal_adv_ccs|coal_adv_ccs|M1,TWh/yr,2726,NaN,NaN,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,in,cooling,cooling__coal_adv_ccs,coal_adv_ccs,M1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9446,MESSAGEix-CA,baseline,Yukon|Yukon,out|water_supply_constraint|water_constraint|e...,TWh/yr,44832,0.0,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,out,water_supply_constraint,water_constraint,extract__water_constraint,M1
9447,MESSAGEix-CA,baseline,Yukon|Yukon,out|water_supply|freshwater_instream|extract__...,TWh/yr,44833,0.0,0.0,0.000015,0.000015,...,0.000015,0.000015,0.000015,0.000015,0.000015,out,water_supply,freshwater_instream,extract__freshwater_instream,M1
9448,MESSAGEix-CA,baseline,Yukon|Yukon,out|water_supply|freshwater_supply|extract__fr...,TWh/yr,44834,0.0,0.0,0.129990,0.182118,...,0.002937,0.017272,0.743716,1.056791,0.009193,out,water_supply,freshwater_supply,extract__freshwater_supply,M1
9449,MESSAGEix-CA,baseline,Yukon|Yukon,out|water_supply|saline_supply|extract__saline...,TWh/yr,44835,0.0,0.0,0.051205,0.000000,...,0.530915,0.536563,0.000000,0.000000,0.808410,out,water_supply,saline_supply,extract__saline_supply,M1


In [5]:
sankey

<class 'pyam.core.IamDataFrame'>
Index:
 * model      : MESSAGEix-CA (1)
 * scenario   : baseline (1)
Timeseries data coordinates:
   region     : Alberta|Alberta, BritishColumbia|BritishColumbia, ... Yukon|Yukon (13)
   variable   : in|cooling|cooling__bio_hpl|bio_hpl|M1, ... (727)
   unit       : TWh/yr (1)
   year       : 2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045, ... 2060 (11)
   unnamed: 0 : 2722, 2723, 2724, 2725, 2726, 2727, 2728, 2729, ... 44836 (9451)

In [6]:

years, regions

(Index([2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060], dtype='int64'),
 ['Alberta|Alberta',
  'BritishColumbia|BritishColumbia',
  'Manitoba|Manitoba',
  'NewBrunswick|NewBrunswick',
  'NewfoundlandandLabrador|NewfoundlandandLabrador',
  'NorthwestTerritories|NorthwestTerritories',
  'NovaScotia|NovaScotia',
  'Nunavut|Nunavut',
  'Ontario|Ontario',
  'PrinceEdwardIsland|PrinceEdwardIsland',
  'Quebec|Quebec',
  'Saskatchewan|Saskatchewan',
  'Yukon|Yukon'])

In [7]:
csvs = []
for year in years:
    for region in regions:
        
        print(year, region)
        
        aux2_csv_regional = aux2_csv[aux2_csv.region == region]
        csv = pd.DataFrame()
        Source = []
        Target = []
        Value = []
        for i, j in aux2_csv_regional.iterrows():
            if j[['flow_type']].item() == 'in':
                source = [j[['level']].item()+'_'+j[['commodity']].item()]
                target = [j[['technology']].item()+'_'+j[['mode']].item()]
                value = [j[[year]].item()]
            elif j[['flow_type']].item() == 'out':
                source = [j[['technology']].item()+'_'+j[['mode']].item()]
                target = [j[['level']].item()+'_'+j[['commodity']].item()]
                value = [j[[year]].item()]
            Source = Source + source
            Target = Target + target
            Value = Value + value
        csv['Source'] = Source
        csv['Target'] = Target
        csv['Value'] = Value
        csv['Time'] = year
        csv['Region'] = region
        
        # Drop flows which are 0 and infintesimal
        csv = csv[csv.Value != 0.0]
        csv = csv[csv.Value >= 0.0001]
        
        # Drop not necessary flows (certainly depends on the model)
        csv = csv[csv.Target != 'emi_dummy_emi_dummy']   
        csv = csv[csv.Source != 'oil_extr_mpen_M1']
        csv = csv[csv.Source != 'gas_extr_mpen_M1']
        csv = csv[csv.Source != 'cement_CO2_M1']
        csv = csv[csv.Source != 'coal_extr_mpen_M1']
        csv = csv[csv.Source != 'flaring_CO2_M1']
        csv = csv[csv.Source != 'co2_tr_dis_M1']
        
        csvs.append(csv)

2010 Alberta|Alberta
2010 BritishColumbia|BritishColumbia
2010 Manitoba|Manitoba
2010 NewBrunswick|NewBrunswick
2010 NewfoundlandandLabrador|NewfoundlandandLabrador
2010 NorthwestTerritories|NorthwestTerritories
2010 NovaScotia|NovaScotia
2010 Nunavut|Nunavut
2010 Ontario|Ontario
2010 PrinceEdwardIsland|PrinceEdwardIsland
2010 Quebec|Quebec
2010 Saskatchewan|Saskatchewan
2010 Yukon|Yukon
2015 Alberta|Alberta
2015 BritishColumbia|BritishColumbia
2015 Manitoba|Manitoba
2015 NewBrunswick|NewBrunswick
2015 NewfoundlandandLabrador|NewfoundlandandLabrador
2015 NorthwestTerritories|NorthwestTerritories
2015 NovaScotia|NovaScotia
2015 Nunavut|Nunavut
2015 Ontario|Ontario
2015 PrinceEdwardIsland|PrinceEdwardIsland
2015 Quebec|Quebec
2015 Saskatchewan|Saskatchewan
2015 Yukon|Yukon
2020 Alberta|Alberta
2020 BritishColumbia|BritishColumbia
2020 Manitoba|Manitoba
2020 NewBrunswick|NewBrunswick
2020 NewfoundlandandLabrador|NewfoundlandandLabrador
2020 NorthwestTerritories|NorthwestTerritories
2020 N

In [8]:
sankey_processed = pd.concat(csvs, axis=0)
sankey_processed

,Source,Target,Value,Time,Region
6,cooling_cooling__coal_ppl,coal_ppl_M1,7.130888,2020,Alberta|Alberta
10,cooling_cooling__gas_cc,gas_cc_M1,37.145307,2020,Alberta|Alberta
11,cooling_cooling__gas_hpl,gas_hpl_M1,4.153225,2020,Alberta|Alberta
20,final_biomass,biomass_i_M1,21.065546,2020,Alberta|Alberta
21,final_biomass,biomass_rc_M1,8.764657,2020,Alberta|Alberta
...,...,...,...,...,...
703,hp_gas_rc_M1,useful_rc_therm,0.283958,2060,Yukon|Yukon
713,elec_trp_M1,useful_transport,0.209970,2060,Yukon|Yukon
717,gas_trp_M1,useful_transport,0.209970,2060,Yukon|Yukon
724,extract__freshwater_supply_M1,water_supply_freshwater_supply,0.009193,2060,Yukon|Yukon


In [9]:
report_csv = report.copy()

report_csv["Model"] = "MESSAGEix-Canada"
report_csv["Scenario"] = "Reference"
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "emis", "Emissions", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "in|final|", "Final Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "out|final|", "Final Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "in|primary|", "Primary Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "out|primary|", "Primary Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "in|secondary|", "Secondary Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "out|secondary|", "Secondary Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "in|renewable|", "Primary Energy|renewable|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "out|renewable|", "Primary Energy|renewable|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "total om cost|", "Total Costs|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "out|useful|", "Useful Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "in|useful|", "Useful Energy|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "CAP_NEW|new capacity|", "Capacity Addition|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "CAP|capacity|", "Capacity|", regex=False
)
report_csv["Variable"] = report_csv["Variable"].str.replace(
    "inv cost|", "Capital Cost|", regex=False
)
report_csv["Region"] = report_csv["Region"].apply(lambda x: x.split("|")[0])

report_csv.columns = report_csv.columns.str.strip()  # Remove any leading/trailing whitespace
report_csv.rename(columns={'Model': 'model', 'Scenario': 'scenario', 'Variable': 'variable'}, inplace=True)

report_csv = report_csv.drop_duplicates(subset=["model", "scenario", "Region", "variable"], keep='first')


pyam_csv = pyam.IamDataFrame(report_csv)

variable = pyam_csv.variable
pyam_csv.aggregate_region(variable, append=True)
pyam_csv.rename({"region": {"World": "Canada"}}, inplace=True)

In [10]:
pyam_csv

<class 'pyam.core.IamDataFrame'>
Index:
 * model      : MESSAGEix-Canada (1)
 * scenario   : Reference (1)
Timeseries data coordinates:
   region     : Alberta, BritishColumbia, Canada, Manitoba, ... Yukon (14)
   variable   : Capacity Addition|LNG_exp, ... vom cost|wind_ppl (3436)
   unit       : '' (1)
   year       : 2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045, ... 2060 (11)
   unnamed: 0 : 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, ... 44836 (44668)

In [11]:
report_csv_melted = report_csv.melt(id_vars=['model', 'Region', 'variable', 'Unit', 'scenario'], var_name='Time', value_name='Value')

In [12]:
sankey_processed.columns = sankey_processed.columns.str.lower()
report_csv_melted.columns = report_csv_melted.columns.str.lower()

In [13]:
sankey_processed['variable'] = 'Sankey|' + sankey_processed['source'] + '->' + sankey_processed['target']

sankey_processed = sankey_processed.drop(columns=['source', 'target'])

In [14]:
sankey_processed['model'] = 'MESSAGEix-Canada'
sankey_processed['unit'] = ''

In [15]:
message_results = pd.concat([sankey_processed, report_csv_melted], axis=0)
message_results["region"] = message_results["region"].apply(lambda x: x.split("|")[0])
message_results.rename({"region": {"World": "Canada"}}, inplace=True)
message_results['scenario'] = report['Scenario'].unique()[0]
message_results

,value,time,region,variable,model,unit,scenario
6,7.130888,2020,Alberta,Sankey|cooling_cooling__coal_ppl->coal_ppl_M1,MESSAGEix-Canada,,baseline
10,37.145307,2020,Alberta,Sankey|cooling_cooling__gas_cc->gas_cc_M1,MESSAGEix-Canada,,baseline
11,4.153225,2020,Alberta,Sankey|cooling_cooling__gas_hpl->gas_hpl_M1,MESSAGEix-Canada,,baseline
20,21.065546,2020,Alberta,Sankey|final_biomass->biomass_i_M1,MESSAGEix-Canada,,baseline
21,8.764657,2020,Alberta,Sankey|final_biomass->biomass_rc_M1,MESSAGEix-Canada,,baseline
...,...,...,...,...,...,...,...
536011,0.000000,2060,Yukon,out|water_supply_constraint|water_constraint|e...,MESSAGEix-Canada,,baseline
536012,0.000002,2060,Yukon,out|water_supply|freshwater_instream|extract__...,MESSAGEix-Canada,,baseline
536013,0.001049,2060,Yukon,out|water_supply|freshwater_supply|extract__fr...,MESSAGEix-Canada,,baseline
536014,0.092221,2060,Yukon,out|water_supply|saline_supply|extract__saline...,MESSAGEix-Canada,,baseline


In [16]:
message_results.to_csv('message_results.csv', index=False)